In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import (
    StandardScaler,
    RobustScaler,
    MinMaxScaler,
    MaxAbsScaler,
    OneHotEncoder,
    LabelEncoder,
    Normalizer,
)
from sklearn.model_selection import cross_val_score,train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression,LogisticRegressionCV,RidgeClassifier,RidgeClassifierCV
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier,AdaBoostClassifier,VotingClassifier,BaggingClassifier,RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier,XGBRFClassifier
from catboost import CatBoostClassifier,CatBoost
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
np.random.seed(42)

In [ ]:
data = pd.read_csv('./train.csv')

In [ ]:
data.head()

In [ ]:
X = data.drop('loss',axis=1)
y = data['loss']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.375)

In [ ]:
pres = [StandardScaler(),RobustScaler(),MinMaxScaler(),MaxAbsScaler(),Normalizer(),LabelEncoder()]

In [ ]:
label_encoder = False

In [ ]:
import wandb
PROJECT_NAME = 'Tabular-Aug-Clf'

In [ ]:
wandb.init(project=PROJECT_NAME,name='baseline')
model = GradientBoostingClassifier()
model.fit(X_train,y_train)
wandb.log({'Acc':model.score(X_train,y_train)})
wandb.log({'Val Acc':model.score(X_test,y_test)})
wandb.finish()

In [ ]:
wandb.init(project=PROJECT_NAME,name='Without Fe')
model = GradientBoostingClassifier()
model.fit(X_train,y_train)
wandb.log({'Acc':model.score(X_train,y_train)})
wandb.log({'Val Acc':model.score(X_test,y_test)})
wandb.finish()
for col in list(data.drop(['id','loss'],axis=1).columns):
    max_num = data[col].quantile(0.99)
    min_num = data[col].quantile(0.05)
    data = data[data[col] < max_num]
    data = data[data[col] > min_num]
wandb.init(project=PROJECT_NAME,name='With Fe')
model = GradientBoostingClassifier()
model.fit(X_train,y_train)
wandb.log({'Acc':model.score(X_train,y_train)})
wandb.log({'Val Acc':model.score(X_test,y_test)})
wandb.finish()

In [ ]:
for pre in pres:
    X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.375)
    if label_encoder is True:
        le = LabelEncoder()
        le.fit(y_train)
        y_train = le.transform(y_train)
    wandb.init(project=PROJECT_NAME,name=f'preproccessing-{pre}-labelencoder-{label_encoder}')
    transformation = pre
    transformation.fit(X_train)
    X_train = transformation.transform(X_train)
    model = GradientBoostingClassifier()
    print('Fitting')
    model.fit(X_train,y_train)
    print('Scoring Train')
    acc = model.score(X_train,y_train)
    wandb.log({'Acc':acc})
    print('Scoring Test')
    acc = model.score(transformation.transform(X_test),y_test)
    wandb.log({'Val Acc':acc})